# Обучение Multi-Branch MLP на неразмеченных данных, с использованием pseudo-labeling

In [8]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import DataModule
from lightning_module import BaseLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 1. Загрузка данных


In [ ]:
data_dir = '../data'

dm = DataModule(
    data_dir=data_dir,
    batch_size=128,
    num_workers=4
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')

Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000


In [ ]:
# Загружаем неразмеченные данные
unlabeled_path = os.path.join(data_dir, 'train_unlabeled.csv')
unlabeled_df = pd.read_csv(unlabeled_path)

X_unlabeled = unlabeled_df.values  # (N, 3072)
print(f'Unlabeled samples: {len(X_unlabeled)}')

Unlabeled samples: 14400


## 2. Анализ дисбаланса классов и вычисление весов


In [17]:
train_labels = dm.train_labeled_dataset.y

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)


Class weights: {np.int64(0): np.float64(1.0738255033557047), np.int64(1): np.float64(0.963855421686747), np.int64(2): np.float64(1.0256410256410255), np.int64(3): np.float64(1.103448275862069), np.int64(4): np.float64(0.9523809523809523), np.int64(5): np.float64(0.935672514619883), np.int64(6): np.float64(0.9523809523809523), np.int64(7): np.float64(1.0596026490066226), np.int64(8): np.float64(0.9248554913294798), np.int64(9): np.float64(1.0457516339869282)}


## 3. Создание модели


In [18]:
model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=256,
    output_dim=dm.n_classes,
    num_blocks=4,
    dropout=0.1,
    combine_mode='concat'
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Model parameters: 4,080,650


## 4. Создание Lightning модуля


In [19]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model = BaseLightningModule(
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)


## 5. Обучение модели


In [20]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
)

trainer = Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=True,
    enable_model_summary=True,
    accelerator='auto',
    devices='auto'
)

trainer.fit(lightning_model, dm)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name    | Type             | Params | Mode 
-----------------------------------------------------
0 | model   | MultiBranchMLP   | 4.1 M  | train
1 | loss_fn | CrossEntropyLoss | 0      | train
2 | metrics | ModuleDict       | 0      | train
-----------------------------------------------------
4.1 M     Trainable params
0         Non-trainable params
4.1 M     Total params
16.323    Total estimated model params size (MB)
70        Modules in train mode
0         Modules in eval mode


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Epoch 99: 100%|██████████| 13/13 [00:26<00:00,  0.48it/s, v_num=4, train_loss_step=11.40, val_loss=32.60, train_loss_epoch=17.40]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 13/13 [00:26<00:00,  0.48it/s, v_num=4, train_loss_step=11.40, val_loss=32.60, train_loss_epoch=17.40]


## 6. Оценка на тестовой выборке


In [21]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')

if best_model_path:
    best_model = BaseLightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        loss_fn=loss_fn,
        optimizer_type='adamw',
        learning_rate=1e-3,
        task_type='multiclass'
    )
else:
    best_model = lightning_model

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')


Loading best model from: /Users/muxa/Library/CloudStorage/OneDrive-Личная/Документы/GitHub/MISIS/DL/HomeTask3/baseline/checkpoints/best_model-epoch=71-val_accuracy=0.2835.ckpt
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1600
Test samples: 4000
Testing DataLoader 0: 100%|██████████| 32/32 [00:00<00:00, 261.64it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.2939999997615814
      test_f1_macro         0.2813318073749542
        test_loss           15.571991920471191
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

=== Финальные результаты на тестовой выборке ===
test_loss: 15.5720
test_accuracy: 0.2940
test_f1_macro: 0.2813


## Pseudo-labeling

In [ ]:
X_unlabeled_tensor = torch.FloatTensor(X_unlabeled).to(lightning_model.device)

model.eval()
with torch.no_grad():
    logits = model(X_unlabeled_tensor)
    probabilities = torch.softmax(logits, dim=1)
    confidence, pseudo_labels = torch.max(probabilities, dim=1)

confidence_threshold = 0.95
mask = confidence > confidence_threshold

X_pseudo = X_unlabeled[mask.cpu().numpy()]
y_pseudo = pseudo_labels[mask].cpu().numpy()

print(f'Псевдометки сгенерированы: {len(y_pseudo)} примеров (порог = {confidence_threshold})')

Псевдометки сгенерированы: 11185 примеров (порог = 0.95)


In [ ]:
labeled_df = pd.read_csv(os.path.join(data_dir, 'train_labeled.csv'))
X_labeled = labeled_df.iloc[:, :-1].values
y_labeled = labeled_df.iloc[:, -1].values

X_combined = np.vstack([X_labeled, X_pseudo])
y_combined = np.hstack([y_labeled, y_pseudo])

print(f'Combined training set: {len(X_combined)} samples ({len(X_labeled)} labeled + {len(X_pseudo)} pseudo-labeled)')


Combined training set: 12785 samples (1600 labeled + 11185 pseudo-labeled)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Создаём новый датасет
combined_dataset = TensorDataset(
    torch.FloatTensor(X_combined),
    torch.LongTensor(y_combined)
)
test_dataset = TensorDataset(
    torch.FloatTensor(dm.test_dataset.X),
    torch.LongTensor(dm.test_dataset.y)
)

combined_dataloader = DataLoader(combined_dataset, batch_size=128, shuffle=True, num_workers=4)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

In [ ]:
# Обновляем loss с весами по комбинированному набору (или оставляем старые)
class_weights = compute_class_weight('balanced', classes=np.arange(dm.n_classes), y=y_combined)
class_weights_tensor = torch.FloatTensor(class_weights).to(lightning_model.device)

lightning_model.loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model.learning_rate = 1e-3  # fine-tuning

trainer_finetune = Trainer(
    max_epochs=50,
    accelerator='auto',
    devices='auto',
    enable_progress_bar=True,
    logger=False,
    enable_checkpointing=False
)

trainer_finetune.fit(
    model=lightning_model,
    train_dataloaders=combined_dataloader
)

final_result = trainer_finetune.test(lightning_model, dataloaders=test_dataloader)[0]
print(f"Final Test Accuracy: {final_result['test_accuracy']:.4f}")
print(f"Final Test F1-macro: {final_result['test_f1_macro']:.4f}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name    | Type             | Params | Mode 
-----------------------------------------------------
0 | model   | MultiBranchMLP   | 4.1 M  | eval 
1 | loss_fn | CrossEntropyLoss | 0      | train
2 | metrics | ModuleDict       | 0      | train
-----------------------------------------------------
4.1 M     Trainable params
0         Non-trainable params
4.1 M     Total params
16.323    Total estimated model params size (MB)
4         Modules in train mode
66        Modules in eval mode


Epoch 49: 100%|██████████| 100/100 [00:01<00:00, 66.25it/s, train_loss_step=9.390, train_loss_epoch=14.60]

`Trainer.fit` stopped: `max_epochs=50` reached.


Testing DataLoader 0: 100%|██████████| 32/32 [00:00<00:00, 264.75it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.33500000834465027
      test_f1_macro         0.3284730911254883
        test_loss            715.0737915039062
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Final Test Accuracy: 0.3350
Final Test F1-macro: 0.3285
